# Practice Lab: RAG Fundamentals (Lesson 8.1)

Module 8 · 8 exercises · Chunking + embeddings + hybrid search + evaluation

Exercises 1, 2, 4, 7, 8 run locally (numpy only).


# Lesson 8.1: RAG Fundamentals — Build It From Scratch

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 4, 7, 8 run **locally** (numpy only). Ex 3, 5, 6 are architecture/design.


In [ ]:
import numpy as np
import hashlib, math, re, json
from collections import Counter


---
## Exercise 1 (Easy): Basic RAG Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib

def chunk_text(text, cs=40, ov=5):
    words = text.split()
    chunks = []
    for i in range(0, len(words), cs - ov):
        c = " ".join(words[i:i+cs])
        if len(c.split()) >= 15: chunks.append({"id": len(chunks), "text": c})
    return chunks

def fake_embed(text, dim=64):
    h = hashlib.sha256(text.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"refund":0,"price":1,"course":2,"live":3,"class":4,"policy":5,"cost":6,"rupees":7,"genai":8,"python":9}
    for w, i in kw.items():
        if w in text.lower().split() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

class SimpleRAG:
    def __init__(self, cs=40, ov=5, k=3):
        self.cs, self.ov, self.k = cs, ov, k
        self.chunks, self.embs = [], []
    def add(self, text, src="doc"):
        for c in chunk_text(text, self.cs, self.ov):
            c["source"] = src
            self.chunks.append(c)
            self.embs.append(fake_embed(c["text"]))
        print(f"  Indexed {len(self.chunks)} chunks from '{src}'")
    def retrieve(self, q, k=None):
        k = k or self.k
        qe = fake_embed(q)
        scores = sorted(enumerate(cosine(qe, e) for e in self.embs), key=lambda x: -x[1])[:k]
        return [{"text": self.chunks[i]["text"][:80], "source": self.chunks[i]["source"], "score": round(s, 4)} for i, s in scores]
    def query(self, q):
        docs = self.retrieve(q)
        top = docs[0]
        if top["score"] < 0.15:
            return {"answer": "I don't have this information.", "score": top["score"]}
        return {"answer": f"Based on {top['source']}: {top['text'][:100]}...", "score": top["score"]}

rag = SimpleRAG()
print("Building RAG:")
for text, src in [
    ("Netsetos is an edtech company in Hyderabad offering GenAI Data Science and Cloud courses flagship GenAI Complete Course 14 modules 146 hours", "about.txt"),
    ("Refund policy Full refund within 7 days 50 percent from 8 to 30 days No refunds after 30 days Processed in 5 business days", "refund.txt"),
    ("GenAI course pricing 14999 rupees one time payment Includes lifetime access Discord weekly live sessions certificate 5000 GCP credits EMI via Razorpay", "pricing.txt"),
    ("Live classes daily 7 PM IST on YouTube Recordings in 2 hours Python and GCP 85 percent completion 92 percent placement 4.8 star rating", "faq.txt"),
]:
    rag.add(text, src)

print(f"\nRAG Q&A:")
for q in ["What is the refund policy?", "How much does GenAI cost?", "When are live classes?", "Does Netsetos offer blockchain?"]:
    r = rag.query(q)
    print(f"  Q: {q}")
    print(f"  A: {r['answer'][:80]}  (score: {r['score']})")


</details>


---
## Exercise 2 (Easy): Chunk Size Experiment


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib

def fake_embed(text, dim=64):
    h = hashlib.sha256(text.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"rag":0,"retrieval":1,"embedding":2,"vector":3,"chunk":4,"search":5,"hybrid":6,"bm25":7}
    for w, i in kw.items():
        if w in text.lower().split() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

corpus = ("Retrieval Augmented Generation or RAG is a technique that combines information retrieval "
          "with language model generation. The key idea is to retrieve relevant documents from a "
          "knowledge base and inject them into the prompt. This prevents hallucination because the "
          "model answers from provided context. The RAG pipeline has five steps chunking embedding "
          "storage retrieval and generation. Chunk size is the most important tuning parameter. "
          "Too small and chunks lack context. Too large and they contain mixed topics. "
          "The Vecta benchmark found recursive splitting at 512 tokens achieved 69 percent accuracy. "
          "Hybrid search combines dense embeddings for semantic matching with BM25 for exact keywords "
          "giving a 26 percent NDCG improvement over dense alone. Reranking with cross encoders "
          "adds another 10 to 25 percent precision improvement. " * 3)

queries = [("What is RAG?", "retrieval augmented"), ("How does chunking work?", "chunk"),
           ("What is hybrid search?", "hybrid search"), ("Best chunk size?", "512 tokens")]

sizes = [32, 64, 128, 256, 512]
print(f"Corpus: {len(corpus.split())} words\n")
print(f"{'Size':<8} {'Chunks':<8} {'AvgWords':<10} {'AvgScore':<10} {'Quality'}")
print("=" * 50)

best_s, best_sc = 0, 0
for cs in sizes:
    ov = max(5, int(cs * 0.15))
    words = corpus.split()
    chunks = []
    for i in range(0, len(words), cs - ov):
        c = " ".join(words[i:i+cs])
        if len(c.split()) >= 10: chunks.append(c)
    embs = [fake_embed(c) for c in chunks]
    total = 0
    for q, exp in queries:
        qe = fake_embed(q)
        scores = sorted([(i, cosine(qe, e)) for i, e in enumerate(embs)], key=lambda x: -x[1])
        total += scores[0][1]
    avg = total / len(queries)
    avgw = sum(len(c.split()) for c in chunks) // max(len(chunks), 1)
    quality = "Too small" if cs < 50 else "OPTIMAL" if 64 <= cs <= 256 else "Too large" if cs > 400 else "Good"
    if avg > best_sc: best_sc, best_s = avg, cs
    print(f"{cs:<8} {len(chunks):<8} {avgw:<10} {avg:<10.4f} {quality}")

print(f"\nBest: {best_s} words (score: {best_sc:.4f})")
print(f"Rules: Q&A=256-512, Technical=512-1024, Overlap=10-20%")


</details>


---
## Exercise 4 (Medium): Hybrid Search + Reranking


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np, hashlib, math
from collections import Counter

chunks = [
    "RAG stands for Retrieval Augmented Generation combining search with LLMs",
    "Vector databases like Chroma and Qdrant store embeddings for similarity search",
    "BM25 is a probabilistic ranking function used in text retrieval systems",
    "Cosine similarity measures the angle between two vectors in embedding space",
    "Chunk size of 512 tokens with overlap is the recommended default setting",
    "Reranking with cross encoders improves precision by 10 to 25 percent",
    "Hybrid search combines dense embeddings with sparse BM25 for best results",
    "The refund policy allows full refund within 7 days of purchase at Netsetos",
    "Live classes are held daily at 7 PM IST on YouTube with recordings available",
    "GenAI course costs 14999 rupees with lifetime access and completion certificate",
]

class BM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.docs = [d.lower().split() for d in docs]
        self.k1, self.b = k1, b
        self.avg_dl = sum(len(d) for d in self.docs) / len(self.docs)
        self.df = Counter()
        for doc in self.docs:
            for w in set(doc): self.df[w] += 1
        self.n = len(self.docs)
    def score(self, query):
        q = query.lower().split()
        scores = []
        for doc in self.docs:
            s, tf_map = 0, Counter(doc)
            for t in q:
                if t not in self.df: continue
                tf = tf_map.get(t, 0)
                idf = math.log((self.n - self.df[t] + 0.5) / (self.df[t] + 0.5) + 1)
                s += idf * tf * (self.k1+1) / (tf + self.k1*(1-self.b+self.b*len(doc)/self.avg_dl))
            scores.append(s)
        return scores

def fake_embed(text, dim=64):
    h = hashlib.sha256(text.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"rag":0,"retrieval":1,"embedding":2,"vector":3,"bm25":5,"hybrid":6,"search":7,"refund":10,"price":11,"course":12}
    for w, i in kw.items():
        if w in text.lower().split() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def dense_rank(q, chunks, k=5):
    qe = fake_embed(q)
    return sorted([(i, np.dot(qe, fake_embed(c))/(np.linalg.norm(qe)*np.linalg.norm(fake_embed(c)))) for i, c in enumerate(chunks)], key=lambda x: -x[1])[:k]

def rrf(rankings, k=60):
    scores = {}
    for r in rankings:
        for rank, (idx, _) in enumerate(r):
            scores[idx] = scores.get(idx, 0) + 1.0/(k+rank+1)
    return sorted(scores.items(), key=lambda x: -x[1])

bm25 = BM25(chunks)
queries = ["What is BM25 ranking?", "How does hybrid search work?", "Refund policy?", "Chunk size?"]

print("Hybrid Search Comparison:")
print("=" * 60)
for q in queries:
    bm25_r = sorted(enumerate(bm25.score(q)), key=lambda x: -x[1])[:5]
    dense_r = dense_rank(q, chunks)
    hybrid = rrf([bm25_r, dense_r])[:3]
    print(f"\n  Q: {q}")
    print(f"    BM25:   [{bm25_r[0][0]}] {chunks[bm25_r[0][0]][:50]}...")
    print(f"    Dense:  [{dense_r[0][0]}] {chunks[dense_r[0][0]][:50]}...")
    print(f"    Hybrid: [{hybrid[0][0]}] {chunks[hybrid[0][0]][:50]}...")

print(f"\nRRF: score = sum(1/(60+rank))")
print(f"Impact: +26% NDCG over dense-only")


</details>


---
## Exercise 7 (Challenge): RAGAS Evaluation Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class RAGAS:
    @staticmethod
    def faithfulness(ans, ctx):
        claims = [c.strip() for c in ans.split(".") if len(c.strip()) > 10]
        if not claims: return 1.0, []
        sup, det = 0, []
        for cl in claims:
            w = set(cl.lower().split())
            cw = set(ctx.lower().split())
            ok = len(w & cw) / max(len(w), 1) > 0.3
            sup += int(ok); det.append((cl[:40], ok))
        return sup / len(claims), det
    @staticmethod
    def ctx_prec(ret, rel):
        return sum(1 for d in ret if d in rel) / max(len(ret), 1)
    @staticmethod
    def ctx_rec(ret, all_rel):
        return sum(1 for d in all_rel if d in ret) / max(len(all_rel), 1)

cases = [
    {"q": "What is RAG?", "ctx": "RAG is Retrieval-Augmented Generation combining search with LLMs.",
     "ans": "RAG is Retrieval-Augmented Generation combining search with LLM generation.", "lbl": "Faithful"},
    {"q": "What is RAG?", "ctx": "RAG combines retrieval with generation.",
     "ans": "RAG was invented by Facebook in 2020 and achieves 95% accuracy.", "lbl": "Hallucinated"},
    {"q": "Cost?", "ctx": "Course costs 14999 rupees with lifetime access.",
     "ans": "The course costs 14999 rupees with lifetime access.", "lbl": "Faithful"},
    {"q": "Refund?", "ctx": "Full refund within 7 days. 50% from 8-30 days.",
     "ans": "Full refund within 7 days. 50% until day 30. After 60 days 25% back.", "lbl": "Partial"},
    {"q": "Chunk?", "ctx": "Vecta benchmark found recursive splitting at 512 tokens achieved 69% accuracy.",
     "ans": "Recommended chunk size is 512 tokens per the Vecta benchmark.", "lbl": "Faithful"},
]

ev = RAGAS()
TH = 0.7
print("RAGAS Evaluation:")
print("=" * 55)
scores = []
for c in cases:
    s, d = ev.faithfulness(c["ans"], c["ctx"])
    scores.append(s)
    st = "PASS" if s >= TH else "FAIL"
    print(f"  [{st}] {c['q']}: {s:.2f} ({c['lbl']})")
    for cl, ok in d:
        print(f"    [{'OK' if ok else '!!'}] {cl}...")

print(f"\nContext: prec={ev.ctx_prec(['d1','d2','d3'],['d1','d3']):.2f} "
      f"rec={ev.ctx_rec(['d1','d2','d3'],['d1','d3','d5']):.2f}")
p = sum(1 for s in scores if s >= TH)
print(f"Summary: {p}/{len(cases)} passed, avg={sum(scores)/len(scores):.2f}")
print(f"Flywheel: failed -> golden dataset -> experiments")


</details>


---
## Exercise 8 (Challenge): Hindi/Indic RAG Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

def classify_pdf(text):
    dev = len(re.findall(r'[\u0900-\u097F]', text))
    lat = len(re.findall(r'[a-zA-Z]', text))
    total = len(text.strip())
    if total < 10: return "scanned", "Sarvam Vision OCR"
    if dev > lat * 0.5: return "unicode_hindi", "pymupdf4llm"
    if dev == 0 and any(c in text for c in "\u00bc\u00bd\u00be"): return "legacy", "Font2Unicode"
    if dev == 0: return "english", "pymupdf4llm"
    return "mixed", "Language detection per para"

tests = [
    ("\u092d\u093e\u0930\u0924 \u0938\u0930\u0915\u093e\u0930", "Gov Hindi"),
    ("Supreme Court of India", "English legal"),
    ("", "Scanned"),
    ("\u092e\u0936\u0940\u0928 learning", "Mixed"),
]
print("Indian PDF Classification:")
for text, desc in tests:
    t, a = classify_pdf(text)
    print(f"  [{t:<14}] {desc}: {a}")

print(f"\nIndic RAG Stack:")
for comp, tool in [("PDF", "Sarvam Vision + pymupdf4llm"), ("Legacy", "Font2Unicode"),
    ("Embeddings", "BGE-M3 (100+ langs)"), ("Generation", "Sarvam tokenizer (1.4-2.1 fertility)"),
    ("Evaluation", "ChrF not BLEU/ROUGE"), ("Legal", "IndianKanoon API (1.4M+ docs)")]:
    print(f"  {comp:<14} {tool}")

print(f"\nTokenization Cost (100M words/month):")
for name, fert in [("GPT-4o Hindi", 1.89), ("Llama 3.3", 4.5), ("Sarvam", 1.4), ("GPT-4o English", 1.16)]:
    tok = 100_000_000 * fert; cost = tok * 2.50 / 1e6
    print(f"  {name:<16} {fert:.2f}x -> {tok/1e6:.0f}M tok -> ${cost:.2f}")

print(f"\nCross-lingual: Hindi-Marathi 27.9% R@1 vs Hindi-Tamil 21.2%")
print(f"Solution: dual retrieval (dense Hindi + BM25 English + RRF)")


</details>


---
## Exercise 3 (Medium): PDF RAG with pymupdf4llm
Architecture/design. See practice-lab-8_1.html.


In [ ]:
# Exercise 3: PDF RAG with pymupdf4llm
pass


---
## Exercise 5 (Medium): Embedding Model Comparison
Architecture/design. See practice-lab-8_1.html.


In [ ]:
# Exercise 5: Embedding Model Comparison
pass


---
## Exercise 6 (Challenge): Multi-Document RAG with Citations
Architecture/design. See practice-lab-8_1.html.


In [ ]:
# Exercise 6: Multi-Document RAG with Citations
pass
